# Follow-Your-Motion on the MiraData pairs
Design: `docs/fym-pairs-design-v1.md`. Rows `fym` (per-clip LoRA, FYM's recorded recipe) and `fym_noguide` (Wan2.1-T2V-1.3B without LoRA, FYM's floor) on every gated row of `benchmark/pairs_manifest.csv`, from each pair's `ref21`, tuned on the reference's own caption, generated with the pair's prompt, seed 0. Outputs are mapped back to 720x480 and dropped into `pairs/inbox/`; `poster_score_colab.ipynb` scores them unchanged.

**A100 + High-RAM, its own runtime.** ~52 min per pair (LoRA tuning); resumable. Needs the pair's ref21 to exist (made by `ditflow_pairs_colab.ipynb` Cell 2). Run cells 0-5, then 6 and leave it running.

In [ ]:
# 0. Paths and pins
DRIVE       = "/content/drive/MyDrive"
PAIRS_D     = f"{DRIVE}/ditflow_transfer/pairs"          # refs/ and inbox/ live here (ditflow_pairs_colab + scoring)
RUNS        = f"{DRIVE}/ditflow_transfer/fym_runs"       # FYM run records: logs, LoRAs, done/failed.json
MODELS      = "/content/models/Wan2.1-T2V-1.3B"          # local: tuning reads the weights several times per clip
FYM         = "/content/FollowYourMotion"
FYM_GIT     = "https://github.com/jnheinrich451-eng/FollowYourMotion.git"; FYM_BRANCH = "davis-bench"
DITFLOW     = "/content/ditflow_repo"                    # read-only: benchmark/pairs_manifest.csv
DIT_BRANCH  = "docs/wan-active-instructions"
WAN_REPO    = "Wan-AI/Wan2.1-T2V-1.3B"
WAN_PIN     = f"{DRIVE}/ditflow/fym_wan_REVISION"         # the SAME revision the DAVIS FYM run used
ALLOW_PROVISIONAL = False   # True once the method confirms the census clips' windows (as in ditflow_pairs_colab)
LOG         = f"{DRIVE}/ditflow_transfer/fym_pairs_run.log"


In [ ]:
# 1. Drive, GPU, RAM -- stops on the wrong runtime rather than partway through a ~50 min tuning
from google.colab import drive
drive.mount("/content/drive")
import subprocess, os
gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True).strip()
ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30
print(f"GPU: {gpu} | RAM: {ram_gb:.0f} GB")
assert "A100" in gpu, "not an A100: pick the A100 accelerator and reconnect"
assert ram_gb > 40, "standard RAM: pick High-RAM (the text encoder alone is ~11 GB in CPU RAM)"


In [ ]:
# 2. Code (FYM + the ditflow manifest) and FYM's dependencies, as in fym_davis_colab
def load_env(path):
    out = {}
    for line in open(path, encoding="utf-8"):
        line = line.split("#", 1)[0].strip()
        if "=" in line:
            k, v = line.split("=", 1); out[k.strip()] = v.strip().strip('"').strip("'")
    return out
token = next(iter(load_env(f"{DRIVE}/secrets/gh_token.env").values()))
for repo, url, br in ((FYM, FYM_GIT, FYM_BRANCH),
                      (DITFLOW, "https://github.com/jnheinrich451-eng/ditflow.git", DIT_BRANCH)):
    auth = url.replace("https://", f"https://x-access-token:{token}@")
    if not os.path.exists(repo):
        subprocess.run(["git", "clone", "-q", "-b", br, auth, repo], check=True)
    subprocess.run(["git", "-C", repo, "pull", "-q", "--ff-only"], check=True)
    print(repo, subprocess.run(["git", "-C", repo, "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())
assert os.path.exists(f"{FYM}/fym_bench/run_pairs.py"), "run_pairs.py missing: commit + push it to the FYM repo's davis-bench branch"
!pip install -q transformers==4.46.2 lightning==2.6.1 peft==0.18.1 modelscope ftfy sentencepiece einops imageio imageio-ffmpeg
!cd {FYM} && python -c "import transformers, diffsynth, diffsynth_collect_attn, peft, lightning; print('imports ok, transformers', transformers.__version__)"


In [ ]:
# 3. Wan2.1-T2V-1.3B at the revision the DAVIS FYM run recorded
import pathlib
from huggingface_hub import HfApi, snapshot_download
pin = pathlib.Path(WAN_PIN)
rev = pin.read_text().strip() if pin.exists() else HfApi().model_info(WAN_REPO).sha
if not pin.exists():
    print("! no DAVIS pin found -- pinning the current revision now"); pin.write_text(rev)
if not os.path.exists(f"{MODELS}/diffusion_pytorch_model.safetensors"):
    snapshot_download(WAN_REPO, revision=rev, local_dir=MODELS)
pathlib.Path(MODELS, "REVISION").write_text(f"{WAN_REPO}@{rev}")
print(f"{WAN_REPO}@{rev}")


In [ ]:
# 4. Dry run: which pairs have a reference, and what is left to do
ARGS = ["--manifest", f"{DITFLOW}/benchmark/pairs_manifest.csv", "--refs", f"{PAIRS_D}/refs",
        "--inbox", f"{PAIRS_D}/inbox", "--runs", RUNS, "--models", MODELS] + (["--allow-provisional"] if ALLOW_PROVISIONAL else [])
r = subprocess.run(["python", "fym_bench/run_pairs.py", *ARGS, "--dry-run"], cwd=FYM, capture_output=True, text=True)
print(r.stdout[-4000:], r.stderr[-2000:])


In [ ]:
# 5. Full run, detached: returns at once and survives a kernel restart or a closed VS Code.
#    Order: manifest order = the method's 5 pairs first, then the census clips. Per pair: floor (~1 min), FYM (~52 min).
alive = subprocess.run(["pgrep", "-f", "fym_bench/[r]un_pairs.py"], capture_output=True, text=True).stdout.split()
if alive:
    print("already running (pid", *alive, ") -- not starting another; watch it with cell 7")
else:
    with open(LOG, "a") as log:
        job = subprocess.Popen(["python", "fym_bench/run_pairs.py", *ARGS], cwd=FYM, stdout=log, stderr=subprocess.STDOUT,
                               stdin=subprocess.DEVNULL, start_new_session=True)
    print("started pid", job.pid, "| log:", LOG, "| now run cell 6 and leave it running")


In [ ]:
# 6. Keep-alive: Colab shuts an idle runtime down even while the detached runner works. Leave this running.
#    Interrupting it is safe (the runner keeps going), but the runtime may then idle out. Keep the PC awake.
import time, collections
from IPython.display import clear_output
def status():
    done = collections.Counter(p.parent.parent.name for p in pathlib.Path(RUNS).glob("*/*/seed*/done.json"))
    failed = collections.Counter(p.parent.parent.name for p in pathlib.Path(RUNS).glob("*/*/seed*/failed.json"))
    tail = pathlib.Path(LOG).read_text(errors="replace").strip().splitlines()[-4:] if pathlib.Path(LOG).exists() else []
    return f"done: {dict(done)} | failed: {dict(failed)}\n" + "\n".join(tail)
t0 = time.time()
while subprocess.run(["pgrep", "-f", "fym_bench/[r]un_pairs.py"], capture_output=True).returncode == 0:
    clear_output(wait=True); print(f"runner alive, {(time.time() - t0) / 3600:.1f} h\n" + status()); time.sleep(120)
clear_output(wait=True); print("runner finished\n" + status())


In [ ]:
# 7. Progress and failures (re-run any time)
import json
print(status())
for f in sorted(pathlib.Path(RUNS).glob("*/*/seed*/failed.json")):
    j = json.loads(f.read_text())
    print(f"\n{j['pair']} {j['row']}: {j['stage']} ({j['reason_code']}: {j['reason']})")
    print("   " + "\n   ".join(j["log_tail"].strip().splitlines()[-6:]))
print("\nin the inbox for scoring:", sorted(p.name for p in pathlib.Path(PAIRS_D, "inbox").glob("*__fym*__seed*.mp4")))
